> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🧩 데이터 통합과 훈련/테스트 분할

**목표**: 여러 표를 하나로 합치고(concat·merge), 합친 표를 입력 X와 정답 y, 훈련 데이터와 테스트 데이터로 나눈다.

> 🔁 **만약에 — 데이터가 여러 파일로 온다면?** 1단원에서는 7쪽을 한 표로 모았지만, 실제로는 표가 여러 파일에 나뉘어 오는 일이 많아요. 1과 2에서 이런 경우를 연습하고, 2에서 합친 깨끗한 341줄을 3에서 나눠요.

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추를 누르거나 **Shift + Enter**를 누르면 그 셀이 실행돼요.
> 맨 위 셀부터 **차례로** 실행하세요. 코드를 고쳐 보고 싶다면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 사본을 만들어 두세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명을 사이트의 의사코드와 한 줄씩 맞춰 읽으면 돼요.

- 판다스만 있으면 돼요.

In [ ]:
import pandas as pd
SITE = 'https://yanghyeokjin33.github.io/Algorithm/practice'

## 1. 세로로 이어 붙이기 — `pd.concat`

`pd.read_html(주소)`는 웹 페이지의 <table>을 곧바로 데이터프레임으로 읽어 주는 지름길이에요. 1단원의 크롤링을 한 줄로 줄인 셈이지요. 쪽마다 따로 받은 표 7개를 위아래로 이어 붙여요.

- read_html은 표들을 리스트로 돌려주기 때문에 [0]으로 첫 번째 표를 꺼내요.
- ignore_index=True를 주면 이어 붙인 뒤 인덱스를 0, 1, 2, …로 새로 매겨요.
- 이 345줄은 1단원에서 모은 표와 같아요. 이 표를 3단원에서 다듬어 깨끗한 341줄을 만들었지요.

In [ ]:
pages = []
for p in range(1, 8):
    t = pd.read_html(f'{SITE}/page{p}.html', encoding='utf-8')[0]   # 그 쪽의 첫 번째 표
    pages.append(t)
print([len(t) for t in pages])

crawl = pd.concat(pages)
print(crawl.index[:3], crawl.index[50:53])     # 인덱스가 0, 1, 2 … 다시 0, 1, 2 … 겹쳐요
crawl = pd.concat(pages, ignore_index=True)    # 인덱스를 0부터 새로
print(crawl.shape)

## 2. 열쇠로 옆에 붙이기 — `pd.merge`

🔁 **만약에** 측정값과 종이 서로 다른 파일로 왔다면 어떻게 할까요? 연습을 위해 3단원의 깨끗한 341줄을 두 파일로 나눠 두었어요. 측정표에는 번호·부리·날개·몸무게가 있고 종 칸은 없어요. 판정표에는 번호와 종만 있어요. 두 표에서 **번호**가 같은 행끼리 짝지어 옆으로 붙여요.

- 측정표에는 종 칸이 없고, 판정표에는 번호와 종만 있어요.
- 판정표는 순서가 뒤섞여 있어요. 그래도 번호(열쇠)만 같으면 같은 펭귄이에요.

In [ ]:
measure = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_measure.csv')
label = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_label.csv')
print('측정표', measure.shape, '/ 판정표', label.shape)
label.head()

- on='번호'가 열쇠예요. 두 표의 순서가 달라도 번호로 짝을 찾아요.
- how='left'는 왼쪽 표의 행을 모두 남기고, 짝이 없으면 NaN으로 채워요.
- 341마리가 모두 짝을 찾았기 때문에 두 방법 모두 341줄이에요. 짝이 없는 경우는 사이트 4-2의 100번 펭귄에서 봤지요.

In [ ]:
df = pd.merge(measure, label, on='번호')               # 기본: 두 표에 다 있는 번호만(inner)
print('합친 표:', df.shape)
print('짝을 못 찾아 빠진 펭귄:', len(measure) - len(df))
left = pd.merge(measure, label, on='번호', how='left') # 측정표는 모두 남기기
print('how=left:', left.shape, '→ 종이 빈 행', left['종'].isnull().sum())
df.head()

## 3. 훈련 데이터와 테스트 데이터로 나누기

2에서 번호로 합친 표 `df`(341줄)를 그대로 나눠요. 먼저 입력 **X**와 정답 **y**로 나눠요. X에는 3-1에서 고른 속성 4개가 여러 열로 들어가서 대문자로, y는 한 열뿐이라 소문자로 써요. 그다음 행을 섞어서 80% : 20%로 나눠요.

- test_size=0.2는 20%를 테스트로 떼어 두라는 뜻이에요.
- random_state=42는 섞는 방법을 고정해서 언제 실행해도 같은 결과가 나오게 해요. 사이트에서 본 "씨앗 고정"과 같아요.
- X와 y는 같은 순서로 섞이기 때문에 짝이 깨지지 않아요.
- 합친 표는 깨끗한 표(penguins_clean.csv)와 줄 순서까지 같아요. 그래서 5단원 노트북에서 깨끗한 표를 불러와 똑같이 나눠도, 훈련 272줄과 테스트 69줄이 똑같이 나와요.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['부리길이', '부리깊이', '날개길이', '몸무게']]
y = df['종']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('훈련:', X_train.shape, ' 테스트:', X_test.shape)
print(y_train.value_counts())

## 🧩 확인해 봐요

- test_size를 0.3으로 바꾸면 테스트 데이터는 몇 행이 될까요?
- `stratify=y`를 넣으면 훈련 데이터와 테스트 데이터의 종 비율이 원래 표와 같아져요. 넣어 보고 `y_test.value_counts()` 결과를 견줘 보세요.